In [1]:
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import soundfile as sf
import pretty_midi
import mir_eval

from scipy.signal import butter, sosfiltfilt

In [2]:
DATA_ROOT = Path("data/raw/IDMT_SMT_BASS_SINGLE_TRACKS")
OUTPUT_ROOT = Path("data/interim/dsp/track_001")

audio_path = DATA_ROOT / "audio" / "001.wav"
xml_path = DATA_ROOT / "annotation" / "001.xml"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

audio, sr = sf.read(audio_path)

print(f"Audio: {audio_path}")
print(f"Sample rate: {sr} Hz")
print(f"Shape: {audio.shape}")
print(f"Duration: {len(audio) / sr:.5f} s")
print(f"Peak amplitude: {np.max(np.abs(audio)):.4f}")

Audio: data/raw/IDMT_SMT_BASS_SINGLE_TRACKS/audio/001.wav
Sample rate: 44100 Hz
Shape: (736279,)
Duration: 16.69567 s
Peak amplitude: 0.0610


## 1. Filtro low-pass controlado

Como primera intervención DSP utilizaremos un filtro Butterworth low-pass de
cuarto orden.

El procesamiento se realizará con `sosfiltfilt`, por lo que el filtrado tendrá
fase cero y no introducirá un desplazamiento sistemático de los onsets.

Esto es especialmente importante porque la evaluación posterior considera una
tolerancia temporal de 50 ms.

Se estudiarán cuatro frecuencias de corte:

- 150 Hz
- 200 Hz
- 300 Hz
- 500 Hz

El audio no será normalizado después del filtrado. De esta forma, la intervención
modificará el contenido espectral sin compensar artificialmente los cambios de
nivel producidos por el filtro.

In [3]:
def lowpass_filter(audio, sr, cutoff_hz, order=4):
    """
    Aplica un filtro Butterworth low-pass de fase cero.

    Parameters
    ----------
    audio : np.ndarray
        Señal de audio mono.

    sr : int
        Frecuencia de muestreo en Hz.

    cutoff_hz : float
        Frecuencia de corte del filtro en Hz.

    order : int
        Orden del filtro Butterworth.

    Returns
    -------
    np.ndarray
        Señal filtrada con la misma longitud que la entrada.
    """

    nyquist_hz = sr / 2

    sos = butter(
        order,
        cutoff_hz / nyquist_hz,
        btype="lowpass",
        output="sos",
    )

    filtered = sosfiltfilt(sos, audio)

    return filtered

In [4]:
cutoffs_hz = [150, 200, 300, 500]

filtered_audio = {}

for cutoff_hz in cutoffs_hz:
    y_filtered = lowpass_filter(
        audio,
        sr,
        cutoff_hz=cutoff_hz,
        order=4,
    )

    filtered_audio[cutoff_hz] = y_filtered

    output_path = (
        OUTPUT_ROOT
        / f"001_lowpass_{cutoff_hz}Hz.wav"
    )

    sf.write(
        output_path,
        y_filtered,
        sr,
        subtype="PCM_16",
    )

    print(
        f"{cutoff_hz:>3} Hz → "
        f"{output_path.name} | "
        f"N={len(y_filtered)} | "
        f"peak={np.max(np.abs(y_filtered)):.4f}"
    )

150 Hz → 001_lowpass_150Hz.wav | N=736279 | peak=0.0441
200 Hz → 001_lowpass_200Hz.wav | N=736279 | peak=0.0542
300 Hz → 001_lowpass_300Hz.wav | N=736279 | peak=0.0604
500 Hz → 001_lowpass_500Hz.wav | N=736279 | peak=0.0612


In [5]:
dsp_audit = []

for cutoff_hz, y_filtered in filtered_audio.items():

    output_path = (
        OUTPUT_ROOT
        / f"001_lowpass_{cutoff_hz}Hz.wav"
    )

    y_check, sr_check = sf.read(output_path)

    dsp_audit.append(
        {
            "cutoff_hz": cutoff_hz,
            "sample_rate_hz": sr_check,
            "n_samples": len(y_check),
            "duration_s": len(y_check) / sr_check,
            "peak_amplitude": np.max(np.abs(y_check)),
            "same_sample_rate": sr_check == sr,
            "same_n_samples": len(y_check) == len(audio),
        }
    )

dsp_audit = pd.DataFrame(dsp_audit)

dsp_audit

,cutoff_hz,sample_rate_hz,n_samples,duration_s,peak_amplitude,same_sample_rate,same_n_samples
0,150,44100,736279,16.695669,0.044098,True,True
1,200,44100,736279,16.695669,0.054169,True,True
2,300,44100,736279,16.695669,0.060425,True,True
3,500,44100,736279,16.695669,0.061218,True,True


## 2. Evaluación comparativa

La cantidad de eventos producidos por Basic Pitch varió de forma no monótona
con la frecuencia de corte.

Por sí sola, una reducción en el número de notas estimadas no puede
interpretarse como una mejora, ya que el filtrado podría eliminar tanto
falsos positivos como notas verdaderas.

Por esta razón, todas las condiciones serán evaluadas contra exactamente
el mismo ground truth utilizando las mismas tolerancias del baseline.

Además de Precision, Recall y F1, cuantificaremos cuántos falsos positivos
corresponden a detecciones una o más octavas por encima de una nota de
referencia cercana en el tiempo.

In [6]:
import csv


def load_idmt_reference(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    transcription = root.find("transcription")

    rows = []

    for event in transcription.findall("event"):
        midi = int(event.findtext("pitch"))
        onset_s = float(event.findtext("onsetSec"))
        offset_s = float(event.findtext("offsetSec"))

        rows.append(
            {
                "midi": midi,
                "note": pretty_midi.note_number_to_name(midi),
                "onset_s": onset_s,
                "offset_s": offset_s,
                "duration_s": offset_s - onset_s,
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values("onset_s")
        .reset_index(drop=True)
    )


reference_001 = load_idmt_reference(xml_path)

print(f"Notas de referencia: {len(reference_001)}")
reference_001.head()

Notas de referencia: 44


,midi,note,onset_s,offset_s,duration_s
0,43,G2,0.00000,0.71717,0.71717
1,36,C2,0.78263,1.04350,0.26087
2,38,D2,1.08550,1.32440,0.23890
3,41,F2,1.32640,1.56530,0.23890
4,34,A#1,1.58770,2.04090,0.45320


In [7]:
def load_basic_pitch_events(csv_path):
    rows = []

    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)

        for row in reader:
            rows.append(
                {
                    "onset_s": float(row[0]),
                    "offset_s": float(row[1]),
                    "midi": int(row[2]),
                    "velocity": int(row[3]),
                }
            )

    df = pd.DataFrame(rows)

    df["note"] = df["midi"].apply(
        pretty_midi.note_number_to_name
    )

    df["duration_s"] = (
        df["offset_s"] - df["onset_s"]
    )

    return (
        df
        .sort_values("onset_s")
        .reset_index(drop=True)
    )

In [8]:
def midi_to_hz(midi_notes):
    midi_notes = np.asarray(midi_notes, dtype=float)

    return 440.0 * 2 ** (
        (midi_notes - 69.0) / 12.0
    )


def evaluate_condition(reference, estimated, condition):
    ref_intervals = reference[
        ["onset_s", "offset_s"]
    ].to_numpy(dtype=float)

    est_intervals = estimated[
        ["onset_s", "offset_s"]
    ].to_numpy(dtype=float)

    ref_pitches_hz = midi_to_hz(reference["midi"])
    est_pitches_hz = midi_to_hz(estimated["midi"])

    # --------------------------------------------------
    # Pitch + onset
    # --------------------------------------------------

    precision, recall, f1, _ = (
        mir_eval.transcription.precision_recall_f1_overlap(
            ref_intervals,
            ref_pitches_hz,
            est_intervals,
            est_pitches_hz,
            onset_tolerance=0.050,
            pitch_tolerance=50.0,
            offset_ratio=None,
        )
    )

    matches = mir_eval.transcription.match_notes(
        ref_intervals,
        ref_pitches_hz,
        est_intervals,
        est_pitches_hz,
        onset_tolerance=0.050,
        pitch_tolerance=50.0,
        offset_ratio=None,
    )

    matched_ref = {r for r, e in matches}
    matched_est = {e for r, e in matches}

    tp = len(matches)
    fp = len(estimated) - tp
    fn = len(reference) - tp

    # --------------------------------------------------
    # Pitch + onset + offset
    # --------------------------------------------------

    p_offset, r_offset, f1_offset, _ = (
        mir_eval.transcription.precision_recall_f1_overlap(
            ref_intervals,
            ref_pitches_hz,
            est_intervals,
            est_pitches_hz,
            onset_tolerance=0.050,
            pitch_tolerance=50.0,
            offset_ratio=0.2,
            offset_min_tolerance=0.050,
        )
    )

    # --------------------------------------------------
    # Diagnóstico de falsos positivos de octava superior
    # --------------------------------------------------

    false_positive_idx = [
        idx
        for idx in range(len(estimated))
        if idx not in matched_est
    ]

    upper_octave_fp = 0
    plus_12_fp = 0
    plus_24_fp = 0

    for idx in false_positive_idx:
        fp_row = estimated.loc[idx]

        nearby_ref = reference[
            (
                reference["onset_s"]
                - fp_row["onset_s"]
            ).abs() <= 0.050
        ].copy()

        if nearby_ref.empty:
            continue

        nearby_ref["onset_error"] = (
            nearby_ref["onset_s"]
            - fp_row["onset_s"]
        ).abs()

        nearest = nearby_ref.loc[
            nearby_ref["onset_error"].idxmin()
        ]

        semitone_difference = int(
            fp_row["midi"] - nearest["midi"]
        )

        if semitone_difference == 12:
            plus_12_fp += 1
            upper_octave_fp += 1

        elif semitone_difference == 24:
            plus_24_fp += 1
            upper_octave_fp += 1

        elif semitone_difference == 36:
            upper_octave_fp += 1

    pct_upper_octave_fp = (
        100 * upper_octave_fp / fp
        if fp > 0
        else 0.0
    )

    return {
        "condition": condition,
        "n_estimated": len(estimated),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "f1_onset": f1,
        "f1_offset": f1_offset,
        "FP_+12": plus_12_fp,
        "FP_+24": plus_24_fp,
        "FP_upper_octave": upper_octave_fp,
        "pct_FP_upper_octave": pct_upper_octave_fp,
    }

In [9]:
conditions = {
    "Original": Path(
        "data/interim/basic_pitch/baseline_001/"
        "001_basic_pitch.csv"
    ),

    "LP 150 Hz": Path(
        "data/interim/basic_pitch/dsp_001_150Hz/"
        "001_lowpass_150Hz_basic_pitch.csv"
    ),

    "LP 200 Hz": Path(
        "data/interim/basic_pitch/dsp_001_200Hz/"
        "001_lowpass_200Hz_basic_pitch.csv"
    ),

    "LP 300 Hz": Path(
        "data/interim/basic_pitch/dsp_001_300Hz/"
        "001_lowpass_300Hz_basic_pitch.csv"
    ),

    "LP 500 Hz": Path(
        "data/interim/basic_pitch/dsp_001_500Hz/"
        "001_lowpass_500Hz_basic_pitch.csv"
    ),
}


evaluation_rows = []

for condition, csv_path in conditions.items():

    estimated = load_basic_pitch_events(
        csv_path
    )

    evaluation_rows.append(
        evaluate_condition(
            reference_001,
            estimated,
            condition,
        )
    )


comparison_001 = pd.DataFrame(
    evaluation_rows
)

comparison_001

,condition,n_estimated,TP,FP,FN,precision,recall,f1_onset,f1_offset,FP_+12,FP_+24,FP_upper_octave,pct_FP_upper_octave
0,Original,94,41,53,3,0.436170,0.931818,0.594203,0.478261,32,7,39,73.584906
1,LP 150 Hz,68,39,29,5,0.573529,0.886364,0.696429,0.553571,21,0,21,72.413793
2,LP 200 Hz,89,40,49,4,0.449438,0.909091,0.601504,0.526316,38,0,38,77.551020
3,LP 300 Hz,104,38,66,6,0.365385,0.863636,0.513514,0.459459,40,4,44,66.666667
4,LP 500 Hz,101,40,61,4,0.396040,0.909091,0.551724,0.455172,38,5,43,70.491803


## 3. Resultado del experimento piloto

En `001.wav`, el filtro low-pass de 150 Hz produjo el mejor compromiso entre
Precision y Recall.

Respecto del baseline sin DSP:

- Precision aumentó de **0.436 a 0.574**;
- Recall disminuyó moderadamente de **0.932 a 0.886**;
- F1 de pitch + onset aumentó de **0.594 a 0.696**;
- F1 de pitch + onset + offset aumentó de **0.478 a 0.554**;
- los falsos positivos disminuyeron de **53 a 29**.

Además, las detecciones falsas asociadas a octavas superiores disminuyeron
de 39 a 21, y los errores de +24 semitonos disminuyeron de 7 a 0.

Las condiciones de 200, 300 y 500 Hz no produjeron una mejora comparable,
y las dos últimas redujeron el F1 respecto del baseline.

Este resultado debe interpretarse como evidencia piloto sobre un único track.
No se ajustarán nuevas frecuencias de corte utilizando `001.wav` antes de
evaluar si el patrón se reproduce en el conjunto completo de 17 grabaciones.

In [10]:
baseline = comparison_001.loc[
    comparison_001["condition"] == "Original"
].iloc[0]

comparison_delta_001 = comparison_001.copy()

comparison_delta_001["delta_precision"] = (
    comparison_delta_001["precision"] - baseline["precision"]
)

comparison_delta_001["delta_recall"] = (
    comparison_delta_001["recall"] - baseline["recall"]
)

comparison_delta_001["delta_f1_onset"] = (
    comparison_delta_001["f1_onset"] - baseline["f1_onset"]
)

comparison_delta_001["delta_f1_offset"] = (
    comparison_delta_001["f1_offset"] - baseline["f1_offset"]
)

comparison_delta_001[
    [
        "condition",
        "delta_precision",
        "delta_recall",
        "delta_f1_onset",
        "delta_f1_offset",
    ]
]

,condition,delta_precision,delta_recall,delta_f1_onset,delta_f1_offset
0,Original,0.000000,0.000000,0.000000,0.000000
1,LP 150 Hz,0.137359,-0.045455,0.102226,0.075311
2,LP 200 Hz,0.013268,-0.022727,0.007301,0.048055
3,LP 300 Hz,-0.070786,-0.068182,-0.080689,-0.018801
4,LP 500 Hz,-0.040131,-0.022727,-0.042479,-0.023088
